# Modelo de Predicción Player + Team Attributes

MODELO XGBOOST

TARGET: Home Win (H), Draw (D), Away Win (A)

FEATURES: Player Attributes + Team Attributes

Combina los dos enfoques desarrollados por separado en `Pred-PA.ipynb` (estadísticas individuales de los 22 jugadores titulares) y `Pred-TA.ipynb` (estadísticas tácticas de ambos equipos) en un único modelo entrenado sobre la unión de ambos conjuntos de features.

In [1]:
# HELPERS
import sqlite3 as sql
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

**Cargamos los Dataframes Necesarios**

Match

In [2]:
conn = sql.connect('../../data/database.sqlite')
query = "SELECT * FROM Match"
df_match = pd.read_sql_query(query, conn)

Se crea una columna Full Time Results (FTR) con las clases Home Win (H), Draw (D), Away Win (A) numerizadas (0, 1 y 2 respectivamente).

In [ ]:
conditions = [
    df_match["home_team_goal"] > df_match["away_team_goal"],
    df_match["home_team_goal"] < df_match["away_team_goal"]
]

choices = [0, 2]

df_match["FTR"] = np.select(conditions, choices, default=1) # type: ignore

Se definen las columnas con proveedores de casas de apuestas para eliminarlas del dataframe de partidos

In [4]:
betting_columns = [
    'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 
    'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 
    'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 
    'SJH', 'SJD', 'SJA', 'VCH', 'VCD', 'VCA', 
    'GBH', 'GBD', 'GBA', 'BSH', 'BSD', 'BSA'
]
df_match = df_match.drop(columns=betting_columns)

Se eliminan el resto de columnas con datos nominales

In [5]:
df_match = df_match.select_dtypes(include="number")

Player_Attributes

In [6]:
conn = sql.connect('../../data/database.sqlite')
query = "SELECT * FROM Player_Attributes"
df_player_attributes = pd.read_sql_query(query, conn)

Se obtiene la última versión actualizada con los datos del jugador. Se eliminan los atributos nominales para quedarnos con los numéricos, es decir sus estadísticas como jugador.

In [7]:
player_attrs = (
    df_player_attributes
    .sort_values("date")
    .drop_duplicates("player_api_id", keep="last")
    .select_dtypes(include="number")
)

Team_Attributes

In [8]:
conn = sql.connect('../../data/database.sqlite')
query = "SELECT * FROM Team_Attributes"
df_team_attributes = pd.read_sql_query(query, conn)

Se obtiene la última versión actualizada con los datos del equipo. Se eliminan los atributos nominales para quedarnos con los numéricos, es decir sus estadísticas tácticas como equipo.

In [9]:
team_attrs = (
    df_team_attributes
    .sort_values("date")
    .drop_duplicates("team_api_id", keep="last")
    .select_dtypes(include="number")
)

Se identifican las columnas del partido que contienen el ID de cada jugador titular (`home_player_1`...`home_player_11`, `away_player_1`...`away_player_11`, excluyendo las columnas `home_player_X*`/`home_player_Y*` de coordenadas de formación) y el ID de cada equipo (`home_team_api_id`, `away_team_api_id`, excluyendo `home_team_goal`/`away_team_goal` por ser la fuente del target).

In [10]:
player_cols = [c for c in df_match.columns if re.fullmatch(r"(home|away)_player_\d+", c)]
team_cols = [c for c in df_match.columns if re.match(r"(home|away)_team", c) and c not in ["home_team_goal", "away_team_goal"]]

print(f"Columnas de jugadores encontradas: {len(player_cols)}")
print(f"Columnas de equipos encontradas: {len(team_cols)}")

Columnas de jugadores encontradas: 22
Columnas de equipos encontradas: 2


Se descartan los identificadores no estadísticos (`id`, `*_fifa_api_id`) de cada dataframe de atributos, conservando solo la clave de unión (`player_api_id` / `team_api_id`) y las columnas numéricas que sí representan estadísticas o atributos del jugador/equipo.

In [11]:
stat_cols_player = [c for c in player_attrs.columns if c not in ("id", "player_fifa_api_id", "player_api_id")]
stat_cols_team = [c for c in team_attrs.columns if c not in ("id", "team_fifa_api_id", "team_api_id")]

Se define una única función genérica que reemplaza a las versiones separadas `get_player_stats` y `get_team_stats` usadas en los notebooks individuales: dado un id_series (jugador o equipo), el dataframe de atributos correspondiente y el nombre de su columna clave, devuelve las estadísticas con las columnas prefijadas según la posición/rol. El `reset_index` antes del merge y la restauración del índice original al final garantizan que el orden de las filas se preserve exactamente, evitando desalineaciones entre partidos y estadísticas.

In [12]:
def get_entity_stats(id_series, attrs_df, id_col, stat_cols, prefix):
    """
    Devuelve las estadisticas numericas mas recientes de la entidad (jugador o equipo)
    identificada en id_series, con las columnas prefijadas segun la posicion/rol
    (ej: home_player_1_overall_rating, home_team_api_id_defencePressure).
    """
    tmp = id_series.reset_index(drop=True).to_frame(name=id_col)
    tmp = tmp.merge(attrs_df[[id_col] + stat_cols], on=id_col, how="left")
    tmp = tmp.drop(columns=id_col)
    tmp.columns = [f"{prefix}_{c}" for c in tmp.columns]
    tmp.index = id_series.index
    return tmp

Se construye el dataframe final de features concatenando `match_api_id`, `FTR`, las estadísticas de los 22 jugadores titulares y las estadísticas tácticas de ambos equipos. No se incluyen `home_team_goal` ni `away_team_goal`: determinan directamente el target y usarlas como feature constituiría data leakage.

In [13]:
player_feature_blocks = [
    get_entity_stats(df_match[col], player_attrs, "player_api_id", stat_cols_player, col)
    for col in player_cols
]
team_feature_blocks = [
    get_entity_stats(df_match[col], team_attrs, "team_api_id", stat_cols_team, col)
    for col in team_cols
]

df_features = pd.concat(
    [df_match[["match_api_id", "FTR"]]] + player_feature_blocks + team_feature_blocks,
    axis=1
)
df_features.shape

(25979, 790)

Se revisa la proporción de valores faltantes por columna. En las columnas de jugador suelen surgir cuando algún titular no tiene registro en `Player_Attributes`; en las columnas de equipo, `buildUpPlayDribbling` concentra más faltantes porque ese atributo se incorporó más tarde a la base de FIFA y no existe para temporadas anteriores a su inclusión.

**Limitación a tener en cuenta:** tanto `player_attrs` como `team_attrs` toman el último registro disponible sin filtrar por la fecha del partido. Para partidos antiguos esto implica usar atributos medidos después de jugado el partido (información del futuro), una forma de data leakage temporal que infla la performance reportada. Una mejora futura sería tomar, para cada partido, el registro de atributos más cercano pero anterior a su fecha.

In [14]:
missing_ratio = df_features.isna().mean().sort_values(ascending=False)
missing_ratio.head(10)

away_player_11_balance           0.071057
away_player_11_sliding_tackle    0.071057
away_player_11_jumping           0.071057
away_player_11_vision            0.071057
away_player_11_agility           0.071057
away_player_11_curve             0.071057
away_player_11_volleys           0.071057
home_player_11_sliding_tackle    0.070942
home_player_11_vision            0.070942
home_player_11_balance           0.070942
dtype: float64

In [15]:
feature_columns = [c for c in df_features.columns if c not in ("match_api_id", "FTR")]

# Se conservan las medianas: son necesarias despues, en inferencia,
# donde no se dispone de una distribucion completa para recalcularlas.
feature_medians = df_features[feature_columns].median()
df_features[feature_columns] = df_features[feature_columns].fillna(feature_medians)

**División en conjuntos de entrenamiento y prueba**

In [16]:
X = df_features[feature_columns]
y = df_features["FTR"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Entrenamiento del modelo XGBoost**

`objective="multi:softprob"` indica clasificación multiclase con salida probabilística sobre las 3 clases (`num_class=3`). `tree_method="hist"` se fija explícitamente porque, al combinar ambos conjuntos de features, la matriz tiene varios cientos de columnas (~800), y el método de histogramas escala mejor que el método exacto por defecto en ese escenario.

In [17]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    tree_method="hist",
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

**Evaluación del modelo**

In [18]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Home Win", "Draw", "Away Win"]))

Accuracy: 0.5194

              precision    recall  f1-score   support

    Home Win       0.54      0.83      0.66      2384
        Draw       0.28      0.02      0.03      1319
    Away Win       0.48      0.46      0.47      1493

    accuracy                           0.52      5196
   macro avg       0.43      0.44      0.39      5196
weighted avg       0.46      0.52      0.44      5196



La matriz de confusión permite ver, fila por fila, en qué clase real (Home/Draw/Away) el modelo acierta o se confunde, y con qué clase predicha confunde más frecuentemente cada caso.

In [19]:
confusion_matrix(y_test, y_pred)

array([[1986,   34,  364],
       [ 911,   24,  384],
       [ 776,   28,  689]])

**Importancia de features**

Permite comparar, dentro de un mismo modelo, cuánto pesan los atributos individuales de jugadores frente a los atributos tácticos de equipo en la predicción final.

In [20]:
importances = pd.Series(model.feature_importances_, index=feature_columns)
importances.sort_values(ascending=False).head(20)

home_player_3_overall_rating     0.009201
away_player_3_overall_rating     0.009006
home_player_9_overall_rating     0.006517
home_player_2_overall_rating     0.006361
away_player_6_overall_rating     0.006264
home_player_10_overall_rating    0.005986
away_player_9_overall_rating     0.005625
home_player_4_overall_rating     0.005532
home_player_1_overall_rating     0.005436
home_player_8_overall_rating     0.005352
away_player_4_overall_rating     0.005343
away_player_1_overall_rating     0.004914
away_player_8_potential          0.004892
home_player_7_overall_rating     0.004678
away_player_7_overall_rating     0.004598
away_player_4_potential          0.004511
away_player_1_gk_diving          0.004447
home_player_1_gk_reflexes        0.004142
away_player_8_overall_rating     0.004115
away_player_5_short_passing      0.003973
dtype: float32

## Persistencia del modelo

Junto con el modelo se guarda todo el contexto necesario para reconstruir features consistentes en una futura inferencia: columnas y orden esperado (`feature_columns`), las columnas estadísticas de cada entidad (`stat_cols_player`, `stat_cols_team`), los slots de jugador/equipo (`player_cols`, `team_cols`) y las medianas de imputación (`feature_medians`).

In [21]:
import joblib

joblib.dump({
    "model": model,
    "feature_columns": feature_columns,
    "stat_cols_player": stat_cols_player,
    "stat_cols_team": stat_cols_team,
    "player_cols": player_cols,
    "team_cols": team_cols,
    "feature_medians": feature_medians,
}, "../../models/modelo_pred_pa_ta.joblib")

['../../models/modelo_pred_pa_ta.joblib']

## Inferencia a partir de IDs

A diferencia del enfoque explorado para `Pred-PA.ipynb` (que resolvía nombre → ID consultando la tabla `Player`), aquí la inferencia toma directamente los IDs (`player_api_id` y `team_api_id`) como entrada, sin ninguna resolución de nombres. Esto simplifica la función y evita la ambigüedad de nombres repetidos, a cambio de que quien llame a `predict_match` deba conocer de antemano los IDs (se pueden obtener una sola vez con una consulta SQL directa a las tablas `Player`/`Team`, fuera de este pipeline).

Sigue aplicando la misma advertencia sobre el orden: el modelo aprendió columnas distintas por slot (`home_player_1_*` no es lo mismo que `home_player_7_*`), así que la posición de cada ID dentro de la lista debe respetar la convención del dataset original (slot 1 = arquero, seguido del resto de la formación).

In [ ]:
def build_match_features(home_player_ids, away_player_ids, home_team_id, away_team_id):
    """Construye el vector de features de un partido a partir de IDs de jugador y de equipo."""
    if len(home_player_ids) != 11 or len(away_player_ids) != 11:
        raise ValueError("Se requieren exactamente 11 IDs de jugador por equipo")

    ids_by_slot = {f"home_player_{i+1}": pid for i, pid in enumerate(home_player_ids)}
    ids_by_slot.update({f"away_player_{i+1}": pid for i, pid in enumerate(away_player_ids)})

    blocks = []
    for slot, pid in ids_by_slot.items():
        row = player_attrs.loc[player_attrs["player_api_id"] == pid, stat_cols_player].reset_index(drop=True)
        row.columns = [f"{slot}_{c}" for c in stat_cols_player] # type: ignore
        blocks.append(row)

    team_ids = {"home_team_api_id": home_team_id, "away_team_api_id": away_team_id}
    for slot, tid in team_ids.items():
        row = team_attrs.loc[team_attrs["team_api_id"] == tid, stat_cols_team].reset_index(drop=True)
        row.columns = [f"{slot}_{c}" for c in stat_cols_team] # type: ignore
        blocks.append(row)

    features = pd.concat(blocks, axis=1)
    # Reordena/completa columnas exactamente como en entrenamiento e imputa con las mismas medianas
    features = features.reindex(columns=feature_columns)
    features = features.fillna(feature_medians)
    return features

In [23]:
def predict_match(home_player_ids, away_player_ids, home_team_id, away_team_id):
    features = build_match_features(home_player_ids, away_player_ids, home_team_id, away_team_id)

    proba = model.predict_proba(features)[0]
    pred_class = model.predict(features)[0]

    labels = ["Home Win", "Draw", "Away Win"]
    print(f"Prediccion: {labels[pred_class]}")
    for label, p in zip(labels, proba):
        print(f"  {label}: {p:.1%}")
    return proba

Ejemplo de uso (reemplazar por los `player_api_id` y `team_api_id` reales; estos pueden obtenerse, por ejemplo, con `SELECT player_api_id FROM Player WHERE player_name = ...` ejecutado una sola vez fuera de este pipeline):

In [24]:
home_player_ids = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111]
away_player_ids = [201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211]
home_team_id = 8634
away_team_id = 8633

predict_match(home_player_ids, away_player_ids, home_team_id, away_team_id)

Prediccion: Home Win
  Home Win: 48.7%
  Draw: 26.3%
  Away Win: 25.0%


array([0.48736987, 0.2629424 , 0.24968775], dtype=float32)